# 09 — Headless Automation: CI/CD & Scripting

**Goal:** Automate software engineering tasks with OpenHands in CI/CD pipelines, scripts, and batch workflows.

**What You'll Learn:**
- Headless mode deep dive: always-approve, JSONL output, exit codes
- GitHub Actions integration patterns
- Parsing JSONL output for downstream processing
- Building a CI pipeline that auto-fixes lint errors
- Batch processing multiple repositories
- Error handling and retry strategies


## 9.1 Headless Mode Recap

Headless mode is the automation workhorse:

```bash
openhands --headless -t "Your task"           # Basic
openhands --headless -f task.txt              # From file
openhands --headless --json -t "Task"         # JSONL output
openhands --headless --override-with-envs -t "Task"  # Use env vars
```

**Critical:** Headless mode ALWAYS auto-approves actions. There is no human-in-the-loop. Run in sandboxed environments.


## 9.2 Exit Codes

OpenHands uses standard exit codes for pipeline integration:

| Code | Meaning | Pipeline Action |
|---|---|---|
| `0` | Task completed successfully | Continue pipeline |
| `1` | Task failed or agent errored | Fail the step |
| `2` | Invalid arguments | Fix configuration |


In [ ]:
# ═══════════════════════════════════════════
# 9.2 Exit Code Handling Pattern
# ═══════════════════════════════════════════
# These are shell patterns for CI/CD pipelines

exit_code_examples = r'''
# ─── Pattern 1: Simple pass/fail ───
openhands --headless -t "Fix all ESLint errors"
if [ $? -eq 0 ]; then
    echo "✓ Lint fixes applied successfully"
else
    echo "✗ Agent failed — check logs"
    exit 1
fi

# ─── Pattern 2: Conditional on exit code ───
openhands --headless -t "Update dependencies"
case $? in
    0) echo "Dependencies updated" ;;
    1) echo "Update failed — creating issue"
       gh issue create --title "Auto-dependency update failed" \
           --body "OpenHands could not complete the update. Check CI logs."
       ;;
    2) echo "Invalid config — check openhands setup" ;;
esac

# ─── Pattern 3: Retry on failure ───
# Some tasks benefit from a retry with a more specific prompt
for attempt in 1 2 3; do
    echo "Attempt $attempt..."
    if openhands --headless -t "Fix all type errors reported by mypy"; then
        echo "✓ Success on attempt $attempt"
        break
    fi
    echo "Attempt $attempt failed, retrying with more context..."
    sleep 10
done
'''
print(exit_code_examples)


## 9.3 JSONL Output — Structured Event Streaming

The `--json` flag streams each event as a JSON line. This is the bridge between OpenHands and other tools.

```json
{"type": "action", "action": "write", "path": "app.py", "content": "..."}
{"type": "observation", "content": "File created successfully"}
{"type": "action", "action": "run", "command": "python app.py"}
{"type": "observation", "content": "Hello World"}
{"type": "finish", "message": "Task completed"}
```


In [ ]:
# ═══════════════════════════════════════════
# 9.3 Parsing JSONL Output
# ═══════════════════════════════════════════
# Parse OpenHands JSONL output for CI/CD reporting

import json

# ─── Sample JSONL (what you'd get from --json flag) ───
sample_jsonl = '''
{"type": "message", "role": "user", "content": "Create hello.py"}
{"type": "action", "action": "write", "path": "hello.py"}
{"type": "observation", "content": "File written: hello.py"}
{"type": "action", "action": "run", "command": "python hello.py"}
{"type": "observation", "content": "Hello, World!"}
{"type": "finish", "message": "Task completed successfully"}
'''

# ─── Parse JSONL line by line ───
# Each line is a complete JSON object — process with standard json.loads()
events = []
for line in sample_jsonl.strip().split('\n'):
    if line.strip():
        events.append(json.loads(line))

# ─── Extract insights from the event stream ───
actions = [e for e in events if e['type'] == 'action']
observations = [e for e in events if e['type'] == 'observation']
finishes = [e for e in events if e['type'] == 'finish']

print(f'Event Summary:')
print(f'  Total events: {len(events)}')
print(f'  Actions taken: {len(actions)}')
for a in actions:
    print(f'    - {a.get("action", "unknown")}: {a.get("command") or a.get("path", "")}')
print(f'  Observations: {len(observations)}')
print(f'  Result: {"SUCCESS" if finishes else "INCOMPLETE"}')
if finishes:
    print(f'    Message: {finishes[0].get("message", "")}')

# ─── In a real CI pipeline, you'd read from a file ───
# with open('output.jsonl') as f:
#     for line in f:
#         event = json.loads(line)
#         # Process event...


## 9.4 GitHub Actions Integration

A complete GitHub Actions workflow that uses OpenHands:


In [ ]:
# ═══════════════════════════════════════════
# 9.4 GitHub Actions Workflow Template
# ═══════════════════════════════════════════
# Save this as .github/workflows/openhands-autofix.yml

github_actions_yaml = '''
name: OpenHands Auto-Fix

on:
  issues:
    types: [labeled]
  workflow_dispatch:
    inputs:
      task:
        description: 'Task for OpenHands'
        required: true

jobs:
  autofix:
    # Only run when the "openhands" label is applied
    if: github.event.label.name == 'openhands'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Setup Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.12'

      - name: Install uv and OpenHands
        run: |
          curl -LsSf https://astral.sh/uv/install.sh | sh
          uv tool install openhands --python 3.12

      - name: Run OpenHands
        env:
          LLM_API_KEY: ${{ secrets.LLM_API_KEY }}
          LLM_MODEL: ${{ vars.LLM_MODEL || 'claude-sonnet-4-5' }}
        run: |
          ISSUE_TITLE="${{ github.event.issue.title }}"
          ISSUE_BODY="${{ github.event.issue.body }}"
          openhands --headless --override-with-envs --json \
            -t "GitHub Issue: $ISSUE_TITLE. Details: $ISSUE_BODY" \
            > openhands_output.jsonl

      - name: Create PR with changes
        if: success()
        run: |
          git config user.name "OpenHands Bot"
          git config user.email "bot@openhands.dev"
          git checkout -b openhands/fix-${{ github.event.issue.number }}
          git add -A
          git diff --staged --quiet || (
            git commit -m "Fix: ${{ github.event.issue.title }}

            Closes #${{ github.event.issue.number }}"
            git push origin openhands/fix-${{ github.event.issue.number }}
            gh pr create \
              --title "Auto-fix: ${{ github.event.issue.title }}" \
              --body "Automated fix by OpenHands. Closes #${{ github.event.issue.number }}"
          )
        env:
          GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}
'''
print(github_actions_yaml)


## 9.5 Batch Processing Multiple Repositories

Use headless mode to apply the same task across many repos:


In [ ]:
# ═══════════════════════════════════════════
# 9.5 Batch Processing Script
# ═══════════════════════════════════════════
# This is a shell script pattern for batch headless runs

batch_script = r'''
#!/bin/bash
# Batch process: add CODEOWNERS to all repos in an org

REPOS=(
    "org/repo-alpha"
    "org/repo-beta"
    "org/repo-gamma"
    "org/repo-delta"
)

TASK="Create a CODEOWNERS file that assigns reviews to the maintainers team"

for repo in "${REPOS[@]}"; do
    echo "=== Processing $repo ==="

    # Clone the repo
    gh repo clone "$repo" "/tmp/batch-$repo"
    cd "/tmp/batch-$repo" || exit 1

    # Run OpenHands
    if openhands --headless --override-with-envs -t "$TASK"; then
        # Commit and create PR if changes were made
        if ! git diff --quiet; then
            git checkout -b openhands/add-codeowners
            git add CODEOWNERS
            git commit -m "Add CODEOWNERS file"
            git push origin openhands/add-codeowners
            gh pr create --title "Add CODEOWNERS" \
                --body "Automated by OpenHands batch processor"
            echo "  -> PR created for $repo"
        else
            echo "  -> No changes needed for $repo"
        fi
    else
        echo "  -> FAILED for $repo — check logs"
    fi

    cd -
done

echo "Batch processing complete"
'''
print(batch_script)


## 9.6 CI/CD Best Practices

1. **Use `--override-with-envs`** with CI secrets — never hardcode keys
2. **Capture JSONL output** for debugging failed runs
3. **Set timeout** on CI steps — agents can loop (GitHub Actions: `timeout-minutes: 30`)
4. **Use a cheaper model for simple tasks** — `gpt-5.4-mini` for lint fixes, `claude-sonnet` for complex refactors
5. **Test headless tasks locally first** — `openhands --headless -t "..."` before pushing to CI
6. **Always create a branch/PR**, never push directly to main
7. **Review diffs before merging** — the agent is fast but not infallible


## Next

→ [10_ide_integration.ipynb](10_ide_integration.ipynb) — Using OpenHands inside JetBrains, VS Code, Zed, and Toad
